<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/CoxPH_AML_subgroup_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q lifelines scikit-survival

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 106.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

Mounted at /content/drive


In [ ]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/combined_survival_model_ready_wout_protocol/"
)

TRAIN_PATH = (
    BASE_DIR /
    "combined_train_survival_model_ready.csv"
)

VALIDATION_PATH = (
    BASE_DIR /
    "combined_validation_survival_model_ready.csv"
)

TEST_PATH = (
    BASE_DIR /
    "combined_test_survival_model_ready.csv"
)

RAW_PATIENT_PATH = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "AML Dataset/aml_target_2018_pub/"
    "data_clinical_patient.txt"
)

OUTPUT_DIR = (
    BASE_DIR /
    "coxph_original_subgroup_performance"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DIR)

/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready_wout_protocol/coxph_original_subgroup_performance


In [ ]:
train_df = pd.read_csv(
    TRAIN_PATH
)

validation_df = pd.read_csv(
    VALIDATION_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

raw_patient_df = pd.read_csv(
    RAW_PATIENT_PATH,
    sep="\t",
    comment="#"
)

print(
    "Train:",
    train_df.shape
)

print(
    "Validation:",
    validation_df.shape
)

print(
    "Test:",
    test_df.shape
)

print(
    "Raw patient:",
    raw_patient_df.shape
)

Train: (620, 68)
Validation: (177, 68)
Test: (89, 68)
Raw patient: (899, 35)


In [ ]:
DURATION_COL = "OS_DAYS"
EVENT_COL = "OS_EVENT"

PATIENT_ID_COL = "PATIENT_ID"
SAMPLE_ID_COL = "SAMPLE_ID"

BEST_PENALIZER = 0.10
BEST_L1_RATIO = 0.50

In [ ]:
EXCLUDED_COLUMNS = [
    PATIENT_ID_COL,
    SAMPLE_ID_COL,
    DURATION_COL,
    EVENT_COL
]

candidate_features = [
    col
    for col in train_df.columns
    if col not in EXCLUDED_COLUMNS
]

candidate_features = [
    feature
    for feature in candidate_features
    if pd.api.types.is_numeric_dtype(
        train_df[feature]
    )
]

zero_variance_features = [
    feature
    for feature in candidate_features
    if (
        train_df[feature]
        .nunique(dropna=True)
        <= 1
    )
]

COX_DROP_COLUMNS = [
    # Reference categories
    "SEX_Female",
    "ETHNICITY_Not Hispanic or Latino",
    "CNS_DISEASE_No",
    "CHLOROMA_No",
    "RACE_White",
    "FAB_NOS",
    "PRIMARY_CYTOGENETIC_CODE_Normal",
    "RISK_GROUP_CLEAN_Standard",
    "CYTOGENETIC_COMPLEXITY_GROUP_0",

    # Redundant cytogenetic variables
    "PRIMARY_CYTOGENETIC_CODE_inv(16)",
    "PRIMARY_CYTOGENETIC_CODE_t(8;21)"
]

ORIGINAL_FEATURES = [
    feature
    for feature in candidate_features
    if (
        feature not in zero_variance_features
        and feature not in COX_DROP_COLUMNS
    )
]

print(
    "Number of CoxPH predictors:",
    len(ORIGINAL_FEATURES)
)

Number of CoxPH predictors: 53


In [ ]:
DEMOGRAPHIC_MODEL_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if (
        feature == "AGE"
        or feature.startswith("SEX_")
        or feature.startswith("RACE_")
    )
]

print(
    "Demographic predictors INCLUDED in CoxPH:"
)

for feature in DEMOGRAPHIC_MODEL_FEATURES:
    print("-", feature)

Demographic predictors INCLUDED in CoxPH:
- AGE
- SEX_Male
- RACE_American Indian or Alaska Native
- RACE_Asian
- RACE_Black or African American
- RACE_Native Hawaiian or other Pacific Islander
- RACE_Other


In [ ]:
def make_cox_dataframe(
    df,
    features
):

    model_df = df[
        features
        +
        [
            DURATION_COL,
            EVENT_COL
        ]
    ].copy()

    for feature in features:

        model_df[feature] = (
            pd.to_numeric(
                model_df[feature],
                errors="raise"
            )
            .astype(float)
        )

    model_df[DURATION_COL] = (
        pd.to_numeric(
            model_df[DURATION_COL],
            errors="raise"
        )
        .astype(float)
    )

    model_df[EVENT_COL] = (
        pd.to_numeric(
            model_df[EVENT_COL],
            errors="raise"
        )
        .astype(int)
    )

    return model_df

In [ ]:
def fit_cox(
    df,
    features
):

    model_df = make_cox_dataframe(
        df,
        features
    )

    model = CoxPHFitter(
        penalizer=BEST_PENALIZER,
        l1_ratio=BEST_L1_RATIO
    )

    model.fit(
        model_df,
        duration_col=DURATION_COL,
        event_col=EVENT_COL,
        show_progress=False
    )

    return model

In [ ]:
def calculate_cindex(
    model,
    df,
    features
):

    if len(df) < 3:
        return np.nan

    events = (
        df[EVENT_COL]
        .astype(bool)
        .to_numpy()
    )

    if events.sum() == 0:
        return np.nan

    X = (
        df[features]
        .astype(float)
    )

    risk = (
        model
        .predict_partial_hazard(X)
        .to_numpy()
        .reshape(-1)
    )

    try:

        cindex = (
            concordance_index_censored(
                events,
                df[DURATION_COL]
                .astype(float)
                .to_numpy(),
                risk
            )[0]
        )

        return cindex

    except Exception:

        return np.nan

In [ ]:
development_df = pd.concat(
    [
        train_df,
        validation_df
    ],
    ignore_index=True
)

print(
    "Development set:",
    development_df.shape
)

print(
    "Test set:",
    test_df.shape
)

Development set: (797, 68)
Test set: (89, 68)


In [ ]:
original_cox = fit_cox(
    development_df,
    ORIGINAL_FEATURES
)

print(
    "Original CoxPH fitted successfully."
)

Original CoxPH fitted successfully.


In [ ]:
overall_test_cindex = (
    calculate_cindex(
        original_cox,
        test_df,
        ORIGINAL_FEATURES
    )
)

print(
    "Overall Test C-index:",
    round(
        overall_test_cindex,
        3
    )
)

Overall Test C-index: 0.684


In [ ]:
demographic_lookup = (
    raw_patient_df[
        [
            "PATIENT_ID",
            "AGE",
            "SEX",
            "RACE"
        ]
    ]
    .drop_duplicates(
        "PATIENT_ID"
    )
    .rename(
        columns={
            "AGE":
                "AGE_ORIGINAL",

            "SEX":
                "SEX_GROUP",

            "RACE":
                "RACE_GROUP"
        }
    )
)

In [ ]:
test_fairness_df = (
    test_df
    .merge(
        demographic_lookup,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Rows before merge:",
    len(test_df)
)

print(
    "Rows after merge:",
    len(test_fairness_df)
)

Rows before merge: 89
Rows after merge: 89


In [ ]:
development_demo_df = (
    development_df
    .merge(
        demographic_lookup,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
_, age_bins = pd.qcut(
    development_demo_df[
        "AGE_ORIGINAL"
    ],
    q=3,
    retbins=True,
    duplicates="drop"
)

age_bins[0] = -np.inf
age_bins[-1] = np.inf

print(
    "Age cutoffs:",
    age_bins
)

Age cutoffs: [-inf   6.  14.  inf]


In [ ]:
_, age_bins = pd.qcut(
    development_demo_df[
        "AGE_ORIGINAL"
    ],
    q=3,
    retbins=True,
    duplicates="drop"
)

age_bins[0] = -np.inf
age_bins[-1] = np.inf

print(
    "Age cutoffs:",
    age_bins
)

Age cutoffs: [-inf   6.  14.  inf]


In [ ]:
# Create age groups using development-set age distribution

development_demo_df = development_df.merge(
    demographic_lookup,
    on="PATIENT_ID",
    how="left",
    validate="one_to_one"
)

# Get tertile boundaries from DEVELOPMENT data
_, age_bins = pd.qcut(
    development_demo_df["AGE_ORIGINAL"],
    q=3,
    retbins=True,
    duplicates="drop"
)

# Make sure all possible test ages are covered
age_bins[0] = -np.inf
age_bins[-1] = np.inf

print("Age bins:", age_bins)

# Apply the development-derived boundaries to TEST data
test_fairness_df["AGE_GROUP"] = pd.cut(
    test_fairness_df["AGE_ORIGINAL"],
    bins=age_bins,
    labels=["Younger", "Middle", "Older"],
    include_lowest=True
)

print("\nAge group successfully created.")

Age bins: [-inf   6.  14.  inf]

Age group successfully created.


In [ ]:
print(test_fairness_df.columns.tolist())

['PATIENT_ID', 'AGE', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE', 'WBC_LOG1P', 'SEX_Female', 'SEX_Male', 'RACE_American Indian or Alaska Native', 'RACE_Asian', 'RACE_Black or African American', 'RACE_Native Hawaiian or other Pacific Islander', 'RACE_Other', 'RACE_White', 'ETHNICITY_Hispanic or Latino', 'ETHNICITY_Not Hispanic or Latino', 'CNS_DISEASE_No', 'CNS_DISEASE_Yes', 'CHLOROMA_No', 'CHLOROMA_Yes', 'FAB_M0', 'FAB_M1', 'FAB_M2', 'FAB_M4', 'FAB_M5', 'FAB_M6', 'FAB_M7', 'FAB_NOS', 'PRIMARY_CYTOGENETIC_CODE_MLL', 'PRIMARY_CYTOGENETIC_CODE_Normal', 'PRIMARY_CYTOGENETIC_CODE_Other', 'PRIMARY_CYTOGENETIC_CODE_inv(16)', 'PRIMARY_CYTOGENETIC_CODE_t(8;21)', 'RISK_GROUP_CLEAN_High', 'RISK_GROUP_CLEAN_Low', 'RISK_GROUP_CLEAN_Missing', 'RISK_GROUP_CLEAN_Standard', 'CYTOGENETIC_COMPLEXITY_GROUP_0', 'CYTOGENETIC_COMPLEXITY_GROUP_1', 'CYTOGENETIC_COMPLEXITY_GROUP_2', 'CYTOGENETIC_COMPLEXITY_GROUP_3', 'CYTOGENETIC_COMPLEXITY_GROUP_4_plus', 'OS_DAYS', 'OS_EVENT', 'SAMP

In [ ]:
print("SEX")
display(
    test_fairness_df[
        "SEX_GROUP"
    ].value_counts()
)

print("AGE")
display(
    test_fairness_df[
        "AGE_GROUP"
    ].value_counts()
)

print("RACE")
display(
    test_fairness_df[
        "RACE_GROUP"
    ].value_counts()
)

SEX


,count
SEX_GROUP,
Male,45
Female,44


AGE


,count
AGE_GROUP,
Younger,35
Middle,32
Older,22


RACE


,count
RACE_GROUP,
White,66
Black or African American,9
Other,7
Asian,4
Unknown,3


In [ ]:
def evaluate_subgroups(
    model,
    df,
    features,
    subgroup_col
):

    results = []

    groups = (
        df[subgroup_col]
        .dropna()
        .unique()
    )

    for group in groups:

        subgroup_df = (
            df.loc[
                df[subgroup_col]
                == group
            ]
            .copy()
        )

        n = len(
            subgroup_df
        )

        n_events = int(
            subgroup_df[
                EVENT_COL
            ].sum()
        )

        n_censored = (
            n
            -
            n_events
        )

        cindex = (
            calculate_cindex(
                model,
                subgroup_df,
                features
            )
        )

        results.append({
            "Group":
                str(group),

            "N":
                n,

            "Events":
                n_events,

            "Censored":
                n_censored,

            "C_index":
                cindex
        })

    return pd.DataFrame(
        results
    )

In [ ]:
sex_results = evaluate_subgroups(
    original_cox,
    test_fairness_df,
    ORIGINAL_FEATURES,
    "SEX_GROUP"
)

print(
    "COXPH — SEX SUBGROUP PERFORMANCE"
)

display(
    sex_results.round(3)
)

COXPH — SEX SUBGROUP PERFORMANCE


,Group,N,Events,Censored,C_index
0,Male,45,18,27,0.645
1,Female,44,14,30,0.714


In [ ]:
age_results = evaluate_subgroups(
    original_cox,
    test_fairness_df,
    ORIGINAL_FEATURES,
    "AGE_GROUP"
)

print(
    "COXPH — AGE SUBGROUP PERFORMANCE"
)

display(
    age_results.round(3)
)

COXPH — AGE SUBGROUP PERFORMANCE


,Group,N,Events,Censored,C_index
0,Older,22,11,11,0.592
1,Younger,35,15,20,0.688
2,Middle,32,6,26,0.706


In [ ]:
race_results = evaluate_subgroups(
    original_cox,
    test_fairness_df,
    ORIGINAL_FEATURES,
    "RACE_GROUP"
)

print(
    "COXPH — RACE SUBGROUP PERFORMANCE"
)

display(
    race_results.round(3)
)

COXPH — RACE SUBGROUP PERFORMANCE


,Group,N,Events,Censored,C_index
0,White,66,23,43,0.715
1,Black or African American,9,5,4,0.533
2,Asian,4,1,3,0.333
3,Other,7,2,5,0.364
4,Unknown,3,1,2,1.000


In [ ]:
sex_final = sex_results.copy()

sex_final.insert(
    0,
    "Subgroup",
    "Sex"
)


age_final = age_results.copy()

age_final.insert(
    0,
    "Subgroup",
    "Age"
)


race_final = race_results.copy()

race_final.insert(
    0,
    "Subgroup",
    "Race"
)


final_subgroup_table = pd.concat(
    [
        sex_final,
        age_final,
        race_final
    ],
    ignore_index=True
)

In [ ]:
print(
    "Overall Test C-index:",
    round(
        overall_test_cindex,
        3
    )
)

display(
    final_subgroup_table.round(3)
)

Overall Test C-index: 0.684


,Subgroup,Group,N,Events,Censored,C_index
0,Sex,Male,45,18,27,0.645
1,Sex,Female,44,14,30,0.714
2,Age,Older,22,11,11,0.592
3,Age,Younger,35,15,20,0.688
4,Age,Middle,32,6,26,0.706
5,Race,White,66,23,43,0.715
6,Race,Black or African American,9,5,4,0.533
7,Race,Asian,4,1,3,0.333
8,Race,Other,7,2,5,0.364
9,Race,Unknown,3,1,2,1.000


In [ ]:
final_subgroup_table[
    "Overall_C_index"
] = overall_test_cindex

final_subgroup_table[
    "Difference_from_Overall"
] = (
    final_subgroup_table[
        "C_index"
    ]
    -
    overall_test_cindex
)

display(
    final_subgroup_table.round(3)
)

,Subgroup,Group,N,Events,Censored,C_index,Overall_C_index,Difference_from_Overall
0,Sex,Male,45,18,27,0.645,0.684,-0.039
1,Sex,Female,44,14,30,0.714,0.684,0.031
2,Age,Older,22,11,11,0.592,0.684,-0.091
3,Age,Younger,35,15,20,0.688,0.684,0.004
4,Age,Middle,32,6,26,0.706,0.684,0.022
5,Race,White,66,23,43,0.715,0.684,0.032
6,Race,Black or African American,9,5,4,0.533,0.684,-0.150
7,Race,Asian,4,1,3,0.333,0.684,-0.350
8,Race,Other,7,2,5,0.364,0.684,-0.320
9,Race,Unknown,3,1,2,1.000,0.684,0.316


In [ ]:
sex_results.to_csv(
    OUTPUT_DIR /
    "coxph_sex_subgroup_performance.csv",
    index=False
)

age_results.to_csv(
    OUTPUT_DIR /
    "coxph_age_subgroup_performance.csv",
    index=False
)

race_results.to_csv(
    OUTPUT_DIR /
    "coxph_race_subgroup_performance.csv",
    index=False
)

final_subgroup_table.to_csv(
    OUTPUT_DIR /
    "coxph_all_subgroup_performance.csv",
    index=False
)

print(
    "Results saved to:"
)

print(
    OUTPUT_DIR
)

Results saved to:
/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready_wout_protocol/coxph_original_subgroup_performance


In [ ]:
# Predict relative risk for every test patient

test_fairness_df["PREDICTED_RISK"] = (
    original_cox
    .predict_partial_hazard(
        test_fairness_df[ORIGINAL_FEATURES].astype(float)
    )
    .to_numpy()
    .reshape(-1)
)

display(
    test_fairness_df[
        [
            "SEX_GROUP",
            "AGE_GROUP",
            "RACE_GROUP",
            "PREDICTED_RISK"
        ]
    ].head()
)

,SEX_GROUP,AGE_GROUP,RACE_GROUP,PREDICTED_RISK
0,Male,Older,White,1.213697
1,Male,Older,White,1.149852
2,Male,Younger,White,0.651388
3,Male,Older,White,0.651389
4,Male,Middle,White,1.148845


In [ ]:
HORIZONS = {
    "1Y": 365,
    "2Y": 730,
    "3Y": 1095
}

X_test_cox = (
    test_fairness_df[
        ORIGINAL_FEATURES
    ]
    .astype(float)
)

survival_functions = (
    original_cox
    .predict_survival_function(
        X_test_cox
    )
)

In [ ]:
def get_survival_at_time(
    survival_functions,
    time_point
):

    times = (
        survival_functions.index
        .to_numpy(dtype=float)
    )

    # Last available baseline time <= requested horizon
    valid_times = times[
        times <= time_point
    ]

    if len(valid_times) == 0:
        return np.full(
            survival_functions.shape[1],
            np.nan
        )

    selected_time = valid_times[-1]

    return (
        survival_functions
        .loc[selected_time]
        .to_numpy(dtype=float)
    )

In [ ]:
for label, horizon in HORIZONS.items():

    test_fairness_df[
        f"SURVIVAL_{label}"
    ] = get_survival_at_time(
        survival_functions,
        horizon
    )

In [ ]:
display(
    test_fairness_df[
        [
            "SURVIVAL_1Y",
            "SURVIVAL_2Y",
            "SURVIVAL_3Y"
        ]
    ].head()
)

,SURVIVAL_1Y,SURVIVAL_2Y,SURVIVAL_3Y
0,0.817819,0.675222,0.629997
1,0.826517,0.689316,0.645497
2,0.897684,0.809961,0.780379
3,0.897684,0.809961,0.780379
4,0.826655,0.689540,0.645745


In [ ]:
print(
    "S1 >= S2:",
    (
        test_fairness_df["SURVIVAL_1Y"]
        >=
        test_fairness_df["SURVIVAL_2Y"]
    ).mean()
)

print(
    "S2 >= S3:",
    (
        test_fairness_df["SURVIVAL_2Y"]
        >=
        test_fairness_df["SURVIVAL_3Y"]
    ).mean()
)

S1 >= S2: 1.0
S2 >= S3: 1.0


In [ ]:
from lifelines import KaplanMeierFitter

In [ ]:
def observed_survival_km(
    subgroup_df,
    horizon
):

    if len(subgroup_df) < 2:
        return np.nan

    kmf = KaplanMeierFitter()

    kmf.fit(
        durations=subgroup_df[DURATION_COL],
        event_observed=subgroup_df[EVENT_COL]
    )

    return float(
        kmf.predict(horizon)
    )

In [ ]:
def subgroup_calibration(
    df,
    subgroup_col
):

    results = []

    for group in (
        df[subgroup_col]
        .dropna()
        .unique()
    ):

        subgroup_df = (
            df.loc[
                df[subgroup_col] == group
            ]
            .copy()
        )

        row = {
            "Group": str(group),
            "N": len(subgroup_df),
            "Events": int(
                subgroup_df[EVENT_COL].sum()
            )
        }

        for label, horizon in HORIZONS.items():

            predicted = (
                subgroup_df[
                    f"SURVIVAL_{label}"
                ]
                .mean()
            )

            observed = (
                observed_survival_km(
                    subgroup_df,
                    horizon
                )
            )

            error = abs(
                predicted - observed
            )

            row[
                f"Predicted_{label}"
            ] = predicted

            row[
                f"Observed_{label}"
            ] = observed

            row[
                f"Error_{label}"
            ] = error

        results.append(row)

    return pd.DataFrame(results)

In [ ]:
sex_calibration = subgroup_calibration(
    test_fairness_df,
    "SEX_GROUP"
)

print("SEX CALIBRATION")

display(
    sex_calibration.round(3)
)

SEX CALIBRATION


,Group,N,Events,Predicted_1Y,Observed_1Y,Error_1Y,Predicted_2Y,Observed_2Y,Error_2Y,Predicted_3Y,Observed_3Y,Error_3Y
0,Male,45,18,0.832,0.909,0.077,0.700,0.676,0.024,0.658,0.605,0.053
1,Female,44,14,0.842,0.792,0.050,0.716,0.722,0.006,0.676,0.697,0.021


In [ ]:
age_calibration = subgroup_calibration(
    test_fairness_df,
    "AGE_GROUP"
)

print("AGE CALIBRATION")

display(
    age_calibration.round(3)
)

AGE CALIBRATION


,Group,N,Events,Predicted_1Y,Observed_1Y,Error_1Y,Predicted_2Y,Observed_2Y,Error_2Y,Predicted_3Y,Observed_3Y,Error_3Y
0,Older,22,11,0.826,0.773,0.053,0.691,0.580,0.111,0.648,0.531,0.117
1,Younger,35,15,0.833,0.824,0.009,0.701,0.618,0.083,0.659,0.559,0.100
2,Middle,32,6,0.848,0.935,0.087,0.727,0.871,0.144,0.688,0.839,0.151


In [ ]:
race_calibration = subgroup_calibration(
    test_fairness_df,
    "RACE_GROUP"
)

print("RACE CALIBRATION")

display(
    race_calibration.round(3)
)

RACE CALIBRATION


,Group,N,Events,Predicted_1Y,Observed_1Y,Error_1Y,Predicted_2Y,Observed_2Y,Error_2Y,Predicted_3Y,Observed_3Y,Error_3Y
0,White,66,23,0.840,0.891,0.051,0.713,0.716,0.003,0.672,0.667,0.005
1,Black or African American,9,5,0.779,0.667,0.112,0.615,0.556,0.059,0.565,0.444,0.120
2,Asian,4,1,0.836,0.750,0.086,0.706,0.750,0.044,0.664,0.750,0.086
3,Other,7,2,0.875,0.857,0.018,0.772,0.714,0.058,0.738,0.714,0.024
4,Unknown,3,1,0.847,0.667,0.181,0.725,0.667,0.058,0.685,0.667,0.019


In [ ]:
def compact_calibration_table(
    calibration_df
):

    return (
        calibration_df[
            [
                "Group",
                "N",
                "Error_1Y",
                "Error_2Y",
                "Error_3Y"
            ]
        ]
        .copy()
    )

In [ ]:
sex_calibration_compact = (
    compact_calibration_table(
        sex_calibration
    )
)

age_calibration_compact = (
    compact_calibration_table(
        age_calibration
    )
)

race_calibration_compact = (
    compact_calibration_table(
        race_calibration
    )
)

print("SEX")
display(
    sex_calibration_compact.round(3)
)

print("AGE")
display(
    age_calibration_compact.round(3)
)

print("RACE")
display(
    race_calibration_compact.round(3)
)

SEX


,Group,N,Error_1Y,Error_2Y,Error_3Y
0,Male,45,0.077,0.024,0.053
1,Female,44,0.050,0.006,0.021


AGE


,Group,N,Error_1Y,Error_2Y,Error_3Y
0,Older,22,0.053,0.111,0.117
1,Younger,35,0.009,0.083,0.100
2,Middle,32,0.087,0.144,0.151


RACE


,Group,N,Error_1Y,Error_2Y,Error_3Y
0,White,66,0.051,0.003,0.005
1,Black or African American,9,0.112,0.059,0.120
2,Asian,4,0.086,0.044,0.086
3,Other,7,0.018,0.058,0.024
4,Unknown,3,0.181,0.058,0.019


In [ ]:
from scipy.stats import (
    mannwhitneyu,
    kruskal
)

In [ ]:
female_risk = (
    test_fairness_df.loc[
        test_fairness_df["SEX_GROUP"]
        == "Female",
        "PREDICTED_RISK"
    ]
    .dropna()
)

male_risk = (
    test_fairness_df.loc[
        test_fairness_df["SEX_GROUP"]
        == "Male",
        "PREDICTED_RISK"
    ]
    .dropna()
)

sex_statistic, sex_p = (
    mannwhitneyu(
        female_risk,
        male_risk,
        alternative="two-sided"
    )
)

print(
    "Mann–Whitney U:",
    round(
        sex_statistic,
        3
    )
)

print(
    "p-value:",
    round(
        sex_p,
        4
    )
)

print(
    "Significant:",
    "Yes"
    if sex_p < 0.05
    else "No"
)

Mann–Whitney U: 931.0
p-value: 0.6312
Significant: No


In [ ]:
age_groups_for_test = []

age_group_names = [
    "Younger",
    "Middle",
    "Older"
]

for group in age_group_names:

    values = (
        test_fairness_df.loc[
            test_fairness_df["AGE_GROUP"]
            == group,
            "PREDICTED_RISK"
        ]
        .dropna()
    )

    if len(values) > 0:
        age_groups_for_test.append(
            values
        )

age_statistic, age_p = kruskal(
    *age_groups_for_test
)

print(
    "Kruskal–Wallis H:",
    round(
        age_statistic,
        3
    )
)

print(
    "p-value:",
    round(
        age_p,
        4
    )
)

print(
    "Significant:",
    "Yes"
    if age_p < 0.05
    else "No"
)

Kruskal–Wallis H: 2.83
p-value: 0.2429
Significant: No


In [ ]:
MIN_GROUP_SIZE = 5

race_counts = (
    test_fairness_df[
        "RACE_GROUP"
    ]
    .value_counts()
)

eligible_races = (
    race_counts[
        race_counts >= MIN_GROUP_SIZE
    ]
    .index
    .tolist()
)

print(
    "Race groups included:",
    eligible_races
)

Race groups included: ['White', 'Black or African American', 'Other']


In [ ]:
race_groups_for_test = []

for group in eligible_races:

    values = (
        test_fairness_df.loc[
            test_fairness_df["RACE_GROUP"]
            == group,
            "PREDICTED_RISK"
        ]
        .dropna()
    )

    if len(values) > 0:

        race_groups_for_test.append(
            values
        )

race_statistic, race_p = (
    kruskal(
        *race_groups_for_test
    )
)

print(
    "Kruskal–Wallis H:",
    round(
        race_statistic,
        3
    )
)

print(
    "p-value:",
    round(
        race_p,
        4
    )
)

print(
    "Significant:",
    "Yes"
    if race_p < 0.05
    else "No"
)

Kruskal–Wallis H: 13.564
p-value: 0.0011
Significant: Yes


In [ ]:
statistical_results = pd.DataFrame(
    [
        {
            "Model":
                "Original CoxPH",

            "Subgroup":
                "Sex",

            "Groups":
                "Female vs Male",

            "Test":
                "Mann–Whitney U",

            "Statistic":
                sex_statistic,

            "p_value":
                sex_p,

            "Significant":
                "Yes"
                if sex_p < 0.05
                else "No"
        },

        {
            "Model":
                "Original CoxPH",

            "Subgroup":
                "Age",

            "Groups":
                "Younger / Middle / Older",

            "Test":
                "Kruskal–Wallis",

            "Statistic":
                age_statistic,

            "p_value":
                age_p,

            "Significant":
                "Yes"
                if age_p < 0.05
                else "No"
        },

        {
            "Model":
                "Original CoxPH",

            "Subgroup":
                "Race",

            "Groups":
                " / ".join(
                    eligible_races
                ),

            "Test":
                "Kruskal–Wallis",

            "Statistic":
                race_statistic,

            "p_value":
                race_p,

            "Significant":
                "Yes"
                if race_p < 0.05
                else "No"
        }
    ]
)

display(
    statistical_results.round(
        {
            "Statistic": 3,
            "p_value": 4
        }
    )
)

,Model,Subgroup,Groups,Test,Statistic,p_value,Significant
0,Original CoxPH,Sex,Female vs Male,Mann–Whitney U,931.000,0.6312,No
1,Original CoxPH,Age,Younger / Middle / Older,Kruskal–Wallis,2.830,0.2429,No
2,Original CoxPH,Race,White / Black or African American / Other,Kruskal–Wallis,13.564,0.0011,Yes
